<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/06_labeling_states.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06_labeling_states.ipynb — Rotulagem Supervisionada (BEFORE / DURING / AFTER)

## Objetivo

Formalizar o ground truth supervisionado em nível de janela (5 minutos), rotulando cada janela temporal como:

- **DURING**: janela pertencente a um episódio crítico detectado no Notebook 04
- **BEFORE**: janela imediatamente anterior ao início de um episódio (contexto)
- **AFTER**: janela imediatamente posterior ao fim de um episódio (estabilização)
- **NORMAL**: janelas fora das três categorias acima

## Contexto metodológico

Os episódios críticos utilizados neste notebook foram definidos no Notebook 04 com base na **taxa de falha**:

\[
fail\_rate = \frac{n\_failed}{n\_events}
\]

A detecção utilizou o limiar estatístico \( \mu + 2\sigma \) aplicado sobre `fail_rate`.

Este notebook **não redefine criticidade**, apenas propaga a segmentação de episódios para rotulagem supervisionada temporal.

## Entradas (artefatos do pipeline)

- `window_5min_features.parquet` (Notebook 05 – features construídas sobre `fail_rate`)
- `episodes_detected.parquet` (Notebook 04 – episódios baseados em taxa)

## Saída (artefato deste notebook)

- `window_5min_labeled.parquet`
- `06_labeling_states_summary.json`

## Estratégia de rotulagem

A base temporal de referência é o eixo `bucket_id` (granularidade de 5 minutos).

Para cada episódio \([start\_bucket, end\_bucket]\):

- DURING recebe as janelas no intervalo fechado.
- BEFORE recebe as \(K_{BEFORE}\) janelas imediatamente anteriores ao início.
- AFTER recebe as \(K_{AFTER}\) janelas imediatamente posteriores ao fim.

## Precedência (para evitar conflitos)

1. DURING
2. BEFORE / AFTER (não sobrescrevem DURING)
3. NORMAL (default)

## Parâmetros

- `K_BEFORE` = 12 (60 minutos)
- `K_AFTER` = 12 (60 minutos)

## Validações de consistência

- Checagem de integridade do eixo temporal
- Checagem de sobreposição entre rótulos
- Distribuição final de classes
- Verificação de que todos os episódios possuem ao menos uma janela DURING
- Verificação de coerência entre número de janelas DURING e total de janelas críticas do NB04

## Observação

A presença de episódios muito próximos pode reduzir a quantidade efetiva de janelas BEFORE/AFTER devido à precedência de DURING.

In [1]:
# ============================================================
# 06_labeling_states.ipynb
# Rotulagem supervisionada BEFORE / DURING / AFTER (5-min)
# Episódios definidos por taxa (fail_rate)
# Pipeline PPCOMP_DM (Google Cluster Trace)
# ============================================================

# -----------------------------
# 0) BOOTSTRAP (Colab + Repo)
# -----------------------------
from pathlib import Path
import os
import sys
import subprocess
import importlib
import random
import numpy as np
import pandas as pd
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Google Drive já montado.")

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    try:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso:", e)

os.chdir(str(REPO_DIR))

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
importlib.invalidate_caches()

from src.paths import FEATURES_PATH, REPORTS_PATH, ensure_dirs
ensure_dirs()

def log(msg):
    print(f"[06_labeling_states] {msg}")

# -----------------------------
# 1) INPUTS
# -----------------------------
FEATURES_FILE = FEATURES_PATH / "window_5min_features.parquet"
EPISODES_FILE = FEATURES_PATH / "episodes_detected.parquet"

assert FEATURES_FILE.exists()
assert EPISODES_FILE.exists()

df = pd.read_parquet(FEATURES_FILE)
episodes = pd.read_parquet(EPISODES_FILE)

assert "bucket_id" in df.columns
assert "fail_rate" in df.columns  # coerência com NB05 novo

df = df.sort_values("bucket_id").reset_index(drop=True)
episodes = episodes.sort_values(["start_bucket", "end_bucket"]).reset_index(drop=True)

log(f"Features shape: {df.shape}")
log(f"Episodes shape: {episodes.shape}")

# -----------------------------
# 2) Parâmetros
# -----------------------------
K_BEFORE = 12
K_AFTER = 12

# -----------------------------
# 3) Inicialização
# -----------------------------
df["state"] = "NORMAL"

during_mask = np.zeros(len(df), dtype=bool)
before_mask = np.zeros(len(df), dtype=bool)
after_mask = np.zeros(len(df), dtype=bool)

# -----------------------------
# 4) DURING
# -----------------------------
for _, ep in episodes.iterrows():
    s = int(ep["start_bucket"])
    e = int(ep["end_bucket"])
    in_range = (df["bucket_id"] >= s) & (df["bucket_id"] <= e)
    during_mask |= in_range.values

df.loc[during_mask, "state"] = "DURING"

log(f"DURING total: {int(during_mask.sum())}")

# -----------------------------
# 5) BEFORE / AFTER
# -----------------------------
for _, ep in episodes.iterrows():
    s = int(ep["start_bucket"])
    e = int(ep["end_bucket"])

    # BEFORE
    before_range = (df["bucket_id"] >= s - K_BEFORE) & (df["bucket_id"] < s)
    before_mask |= (before_range.values & ~during_mask)

    # AFTER
    after_range = (df["bucket_id"] > e) & (df["bucket_id"] <= e + K_AFTER)
    after_mask |= (after_range.values & ~during_mask)

df.loc[(df["state"] == "NORMAL") & before_mask, "state"] = "BEFORE"
df.loc[(df["state"] == "NORMAL") & after_mask, "state"] = "AFTER"

# -----------------------------
# 6) Validações
# -----------------------------
dist = df["state"].value_counts().to_dict()

conf_before_during = int((before_mask & during_mask).sum())
conf_after_during = int((after_mask & during_mask).sum())

episodes_without_during = 0
for _, ep in episodes.iterrows():
    s = int(ep["start_bucket"])
    e = int(ep["end_bucket"])
    hits = int(((df["bucket_id"] >= s) &
                (df["bucket_id"] <= e) &
                (df["state"] == "DURING")).sum())
    if hits == 0:
        episodes_without_during += 1

log(f"Distribuição: {dist}")
log(f"Conflitos BEFORE∩DURING: {conf_before_during}")
log(f"Conflitos AFTER∩DURING: {conf_after_during}")
log(f"Episódios sem DURING: {episodes_without_during}")

# -----------------------------
# 7) Persistência
# -----------------------------
OUT_FILE = FEATURES_PATH / "window_5min_labeled.parquet"
df.to_parquet(OUT_FILE, compression="snappy", index=False)

summary = {
    "rows": int(len(df)),
    "k_before": K_BEFORE,
    "k_after": K_AFTER,
    "states_distribution": {k: int(v) for k, v in dist.items()},
    "episodes_total": int(len(episodes)),
    "episodes_without_during": int(episodes_without_during),
    "conflicts_before_during": int(conf_before_during),
    "conflicts_after_during": int(conf_after_during),
}

summary_file = REPORTS_PATH / "06_labeling_states_summary.json"
summary_file.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

log("Notebook 06 finalizado com sucesso.")
df.head()

Mounted at /content/drive
[06_labeling_states] Features shape: (8914, 27)
[06_labeling_states] Episodes shape: (285, 13)
[06_labeling_states] DURING total: 394
[06_labeling_states] Distribuição: {'NORMAL': 5533, 'BEFORE': 1996, 'AFTER': 991, 'DURING': 394}
[06_labeling_states] Conflitos BEFORE∩DURING: 0
[06_labeling_states] Conflitos AFTER∩DURING: 0
[06_labeling_states] Episódios sem DURING: 0
[06_labeling_states] Notebook 06 finalizado com sucesso.


,bucket_id,bucket_start_us,n_events,n_failed,n_machines,n_collections,mean_priority,mean_req_cpus,mean_req_mem,req_cpus_presence_rate,...,fail_rate,is_critical,lag_1,lag_2,lag_3,rolling_mean_1h,rolling_std_1h,pct_change,zscore_global,state
0,15,4500000000,44,10,41,29,249.522727,0.008850,0.003220,1.0,...,0.227273,0,0.323529,0.282609,0.240000,0.268353,0.043738,-0.297521,0.218397,NORMAL
1,16,4800000000,22,5,22,19,219.045455,0.011749,0.003593,1.0,...,0.227273,0,0.227273,0.323529,0.282609,0.260137,0.042099,0.000000,0.218397,NORMAL
2,17,5100000000,28,5,27,22,279.392857,0.010309,0.004613,1.0,...,0.178571,0,0.227273,0.227273,0.323529,0.246543,0.050266,-0.214286,-0.127306,NORMAL
3,18,5400000000,29,7,29,22,259.517241,0.007249,0.009697,1.0,...,0.241379,0,0.178571,0.227273,0.227273,0.245805,0.045928,0.351724,0.318531,NORMAL
4,19,5700000000,36,9,34,27,237.027778,0.010176,0.009284,1.0,...,0.250000,0,0.241379,0.178571,0.227273,0.246329,0.042547,0.035714,0.379725,NORMAL


## Achados do Notebook 06 — Rotulagem Supervisionada (base: taxa)

### Estrutura do dataset rotulado
- Total de janelas: **8914**
- Total de episódios detectados: **285**
- Total de janelas DURING: **394**
- Nenhum episódio ficou sem janela DURING associada.
- Não houve conflitos entre DURING e BEFORE/AFTER.
- A precedência definida (DURING > BEFORE/AFTER > NORMAL) foi aplicada corretamente.

### Distribuição final das classes

- NORMAL: 5533
- BEFORE: 1996
- AFTER: 991
- DURING: 394

Proporções aproximadas:
- DURING: ~4,42%
- BEFORE: ~22,39%
- AFTER: ~11,12%
- NORMAL: ~62,07%

Observações:
A migração do critério de volume absoluto para taxa resultou em maior número de episódios e aumento da densidade de janelas DURING. O contexto BEFORE expandiu-se significativamente, refletindo maior sensibilidade na detecção de transições operacionais.

### Consistência temporal

- A rotulagem respeita integralmente o eixo `bucket_id`.
- Não há vazamento de informação futura na definição de BEFORE.
- AFTER representa janelas de estabilização pós-evento.
- A causalidade temporal foi preservada.

### Estrutura supervisionada final

A coluna `state` formaliza o ground truth supervisionado e permite:

**1. Classificação binária**
- NORMAL vs DURING
- NORMAL vs (BEFORE + DURING + AFTER)

**2. Classificação multiclasse**
- NORMAL
- BEFORE
- DURING
- AFTER

O problema deixa de ser puramente binário e passa a modelar explicitamente a dinâmica pré e pós-degradação.

### Considerações metodológicas

A estratégia adotada:

- Mantém reprodutibilidade
- Preserva granularidade de 5 minutos
- Evita conflitos entre rótulos
- Permite controle explícito do horizonte de contexto (K_BEFORE e K_AFTER)
- Mantém coerência integral com a definição de episódio baseada em `fail_rate`

O dataset `window_5min_labeled.parquet` consolida a transição entre detecção estatística por taxa e modelagem supervisionada estruturada.